In [ ]:
import random
import copy
import pandas as pd

# --- 1. Classes & Models ---
class Course:
    def __init__(self, cid, name, instructor, students):
        self.cid, self.name, self.instructor, self.students = cid, name, instructor, students

class Room:
    def __init__(self, rid, name, capacity):
        self.rid, self.name, self.capacity = rid, name, capacity

class IndividualSchedule:
    def __init__(self, courses, rooms, timeslots):
        self.courses, self.rooms, self.timeslots = courses, rooms, timeslots
        self.assignments = {c.cid: (random.choice(rooms), random.choice(timeslots)) for c in courses}
        self.fitness = 0

    def calculate_fitness(self):
        clashes, cap_violations, inst_clashes = 0, 0, 0
        sched = [{'cid': c, 'inst': next(x.instructor for x in self.courses if x.cid == c),
                  'stud': next(x.students for x in self.courses if x.cid == c),
                  'rid': r.rid, 'cap': r.capacity, 'ts': ts}
                 for c, (r, ts) in self.assignments.items()]

        for i, a1 in enumerate(sched):
            if a1['stud'] > a1['cap']: cap_violations += 1
            for j in range(i + 1, len(sched)):
                a2 = sched[j]
                if a1['ts'] == a2['ts']:
                    if a1['rid'] == a2['rid']: clashes += 1
                    if a1['inst'] == a2['inst']: inst_clashes += 1

        self.fitness = 1 / (1 + (clashes * 10) + (inst_clashes * 10) + (cap_violations * 5))
        return self.fitness

# --- 2. Genetic Operators ---
def crossover(p1, p2):
    c1, c2 = IndividualSchedule(p1.courses, p1.rooms, p1.timeslots), IndividualSchedule(p2.courses, p2.rooms, p2.timeslots)
    for cid in p1.assignments.keys():
        use_p1 = random.random() < 0.5
        c1.assignments[cid] = p1.assignments[cid] if use_p1 else p2.assignments[cid]
        c2.assignments[cid] = p2.assignments[cid] if use_p1 else p1.assignments[cid]
    return c1, c2

def mutate(ind, rate=0.15):
    for cid in ind.assignments.keys():
        if random.random() < rate:
            if random.random() < 0.5:
                ind.assignments[cid] = (random.choice(ind.rooms), ind.assignments[cid][1])
            else:
                ind.assignments[cid] = (ind.assignments[cid][0], random.choice(ind.timeslots))

# --- 3. Run Optimization ---
def run_ga(courses, rooms, timeslots, pop_size=80, gens=150, mut_rate=0.15):
    pop = [IndividualSchedule(courses, rooms, timeslots) for _ in range(pop_size)]
    for g in range(gens):
        for ind in pop: ind.calculate_fitness()
        pop.sort(key=lambda x: x.fitness, reverse=True)
        if pop[0].fitness == 1.0:
            break

        next_gen = copy.deepcopy(pop[:max(1, int(pop_size * 0.1))])
        while len(next_gen) < pop_size:
            p1, p2 = max(random.sample(pop, 5), key=lambda x: x.fitness), max(random.sample(pop, 5), key=lambda x: x.fitness)
            ch1, ch2 = crossover(p1, p2)
            mutate(ch1, mut_rate); mutate(ch2, mut_rate)
            next_gen.extend([ch1, ch2])
        pop = next_gen[:pop_size]

    pop[0].calculate_fitness()
    return pop[0]

# --- 4. Setup & Execute ---
rooms_data = [Room("R1", "Hall A", 100), Room("R2", "Lab B", 50), Room("R3", "Room C", 30)]
courses_data = [
    Course("CS1", "Intro CS", "Dr. A", 80), Course("CS2", "Data Struct", "Dr. B", 40),
    Course("CS3", "Algorithms", "Dr. A", 45), Course("MA1", "Calculus", "Dr. C", 25)
]
tslists = ["Mon 09:00", "Mon 11:00", "Wed 09:00", "Wed 11:00"]

best = run_ga(courses_data, rooms_data, tslists)

# Display
output = [{"Course": next(c.name for c in courses_data if c.cid == cid), "Instructor": next(c.instructor for c in courses_data if c.cid == cid),
           "Room": r.name, "Timeslot": ts} for cid, (r, ts) in best.assignments.items()]
display(pd.DataFrame(output))

,Course,Instructor,Room,Timeslot
0,Intro CS,Dr. A,Hall A,Mon 09:00
1,Data Struct,Dr. B,Lab B,Mon 11:00
2,Algorithms,Dr. A,Lab B,Wed 11:00
3,Calculus,Dr. C,Lab B,Wed 09:00


In [ ]:
# Run the genetic algorithm and verify that we have zero instructor clashes
best_schedule = run_ga(courses_data, rooms_data, tslists, pop_size=100, gens=200)

# Calculate final metrics
best_schedule.calculate_fitness()

# Verify instructor conflicts manually for verification
sched = [{'cid': cid,
          'name': next(x.name for x in best_schedule.courses if x.cid == cid),
          'inst': next(x.instructor for x in best_schedule.courses if x.cid == cid),
          'room': r.name,
          'ts': ts}
         for cid, (r, ts) in best_schedule.assignments.items()]

# Find clashes
instructor_clashes_found = []
for i, a1 in enumerate(sched):
    for j in range(i + 1, len(sched)):
        a2 = sched[j]
        if a1['ts'] == a2['ts'] and a1['inst'] == a2['inst']:
            instructor_clashes_found.append((a1['name'], a2['name'], a1['inst'], a1['ts']))

# Present results
print(f"--- Optimization Status ---")
print(f"Best Individual Fitness: {best_schedule.fitness:.4f}")
print(f"Instructor Clashes Found: {len(instructor_clashes_found)}")
if len(instructor_clashes_found) > 0:
    for clash in instructor_clashes_found:
        print(f"  - CLASH: '{clash[0]}' and '{clash[1]}' taught by {clash[2]} are both at {clash[3]}")
else:
    print("  - Success! No instructor has to teach 2 classes at the same time.")

print("\n--- Generated Schedule ---")
df_schedule = pd.DataFrame(sched)
display(df_schedule)

--- Optimization Status ---
Best Individual Fitness: 1.0000
Instructor Clashes Found: 0
  - Success! No instructor has to teach 2 classes at the same time.

--- Generated Schedule ---


,cid,name,inst,room,ts
0,CS1,Intro CS,Dr. A,Hall A,Mon 09:00
1,CS2,Data Struct,Dr. B,Lab B,Mon 11:00
2,CS3,Algorithms,Dr. A,Lab B,Wed 11:00
3,MA1,Calculus,Dr. C,Hall A,Mon 11:00
